# Business Forecasting — Tutorial 1
## Time Series Graphics, Moving Averages, and Decomposition in Python

This notebook is designed as a **student reference and a guided tutorial**. It is intentionally more detailed than a normal coding notebook.

For every major task, you will see four things:

1. **What we are trying to do** — the business/statistical purpose.
2. **Why we do it** — what information the method gives us.
3. **Python code with line-by-line comments** — so you can understand what each line does.
4. **How to interpret the result** — because producing a graph is not the same as understanding it.

### Learning objectives
By the end of this tutorial, you should be able to:

- open CSV files in Google Colab;
- import the Python packages needed for basic forecasting work;
- inspect and prepare time-series data;
- create and interpret scatter plots and grouped scatter plots;
- create and interpret a scatterplot matrix;
- create line/time plots and grouped time plots;
- build seasonal plots;
- calculate trailing, centered, weighted, and `2×m` moving averages;
- decompose a time series into trend, seasonal, and remainder components;
- produce a seasonally adjusted series;
- start a **fresh Python notebook** and know which setup steps you need to repeat.

> **Best way to use this notebook:** read the explanation first, run the code cell, study the output, and then read the interpretation immediately below it.

# Part 0 — How a Python analysis starts

Before analyzing data, we need to load the Python packages that provide the functions we will use.

A **package** is a collection of pre-written Python tools. Instead of programming everything from scratch, we import packages that already know how to read data, calculate statistics, create graphs, and analyze time series.

### Packages used in this tutorial

| Package / function | Why we need it |
|---|---|
| `pathlib.Path` | Works with file and folder locations in a clean way. |
| `zipfile` | Opens a ZIP file if the tutorial data are uploaded together. |
| `numpy` | Numerical calculations and arrays; we use it for grouping and weighted averages. |
| `pandas` | Reads CSV files and works with tables, dates, columns, and time series. |
| `matplotlib.pyplot` | Creates graphs. |
| `scatter_matrix` | Creates many pairwise scatter plots at once. |
| `seasonal_decompose` | Separates a time series into trend, seasonal, and remainder components. |

### Important idea for later
Most Python notebooks begin with an **import/setup cell** like the one below. You do not need to memorize every command immediately, but you should recognize what each package is doing.

In [ ]:
# Import Path so Python can work with file and folder locations.
from pathlib import Path

# Import zipfile so Python can open a ZIP archive containing several CSV files.
import zipfile

# Import NumPy and give it the standard short name np for numerical calculations.
import numpy as np

# Import pandas and give it the standard short name pd for working with tables and time-series data.
import pandas as pd

# Import Matplotlib's plotting module and give it the standard short name plt for creating graphs.
import matplotlib.pyplot as plt

# Import scatter_matrix so we can display many pairwise variable relationships in one figure.
from pandas.plotting import scatter_matrix

# Try to import the decomposition function used later in the tutorial.
try:
    # Import seasonal_decompose from statsmodels when the package is already available.
    from statsmodels.tsa.seasonal import seasonal_decompose
except ImportError:
    # Install statsmodels quietly only if it is missing from the current Python environment.
    %pip -q install statsmodels
    # Import seasonal_decompose after the installation finishes.
    from statsmodels.tsa.seasonal import seasonal_decompose

# Set a comfortable default graph size so later plots are easy to read.
plt.rcParams['figure.figsize'] = (10, 5.5)

# Turn on graph grid lines by default because they help us compare values across time.
plt.rcParams['axes.grid'] = True

# Print a confirmation message so we know the setup cell completed successfully.
print('Python setup complete.')

### How Python comments work

A line beginning with `#` is a **comment**. Python does not execute the comment; it is written for the human reader.

In this tutorial, the comments are intentionally detailed. Read the comment first and then the line directly below it. This helps you connect the English explanation to the Python instruction.

Example:

```python
# Read the CSV file and store the table in a DataFrame called df.
df = pd.read_csv('file.csv')
```

As you become more comfortable with Python, your own comments can become shorter. For now, use them as a learning guide.

### What just happened?

The setup cell did **not** analyze any data yet. It only prepared our Python environment.

The most important imports to remember for this course are:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
```

You will see these repeatedly. `pandas` handles the data table, `NumPy` helps with numerical work, and `Matplotlib` creates graphs.

The decomposition function comes from `statsmodels`, which contains many statistical and time-series tools.

# Part 1 — Data files and the Google Colab workflow

This tutorial uses five CSV files:

- `insurance.csv` — advertising and insurance quote data;
- `hourly_weather.csv` — hourly weather variables;
- `apple_stock_daily.csv` — daily stock prices;
- `quarterly_arrivals.csv` — quarterly arrival data for several countries;
- `aus_airpassengers.csv` — an additional annual passenger series used for practice.

## What a CSV file is
A CSV file stores a rectangular data table in plain text. Each row is an observation and each column is a variable.

## What the next code cell does
The next cell tries to find the files automatically. If you are using Google Colab and the files are not already available, it asks you to upload either:

- `Tutorial1_Data.zip`, or
- the individual CSV files.

Once the files are found, the rest of the notebook can use them without repeatedly asking for file locations.

In [ ]:
# Create a dictionary that connects a short internal name to each required CSV filename.
EXPECTED_FILES = {
    'insurance': 'insurance.csv',           # Use this file for the advertising-versus-quotes scatter example.
    'weather': 'hourly_weather.csv',        # Use this file for grouped and pairwise weather relationships.
    'apple': 'apple_stock_daily.csv',       # Use this file for daily time plots and moving averages.
    'arrivals': 'quarterly_arrivals.csv',   # Use this file for quarterly seasonality and decomposition.
    'passengers': 'aus_airpassengers.csv',  # Use this file for an additional student practice exercise.
}

# List the common folders where the notebook may find uploaded or locally stored data files.
SEARCH_FOLDERS = [
    Path.cwd(),                               # Check the notebook's current working folder first.
    Path('/content'),                        # Check Colab's main working folder.
    Path('/content/Tutorial1_Data'), # Check a tutorial-data folder if the ZIP was extracted there.
    Path('/mnt/data'),                       # Check the working folder used when this notebook was tested.
    Path('/mnt/data/Tutorial1_Data'),# Check the tested tutorial-data subfolder.
]

# Define a reusable function that searches the folders above for one filename.
def find_file(filename):
    # Check each possible folder one at a time.
    for folder in SEARCH_FOLDERS:
        # Combine the folder location with the filename we are looking for.
        candidate = folder / filename
        # Test whether this exact file exists in the current folder.
        if candidate.exists():
            # Return the matching location immediately so the search can stop.
            return candidate
    # Return None when the file could not be found in any known location.
    return None

# Build a list containing the filenames that are currently missing.
missing_files = [filename for filename in EXPECTED_FILES.values() if find_file(filename) is None]

# Run the upload instructions only when at least one required file is missing.
if missing_files:
    # Try to use Google Colab's interactive file-upload tool.
    try:
        # Import the Colab file uploader only when it is needed.
        from google.colab import files
        # Tell the student exactly what can be uploaded.
        print('Upload Tutorial1_Data.zip OR the individual tutorial CSV files.')
        # Open the upload dialog and store the uploaded file contents in memory.
        uploaded = files.upload()
        # Process every uploaded file one at a time.
        for uploaded_name, uploaded_bytes in uploaded.items():
            # Create a file location inside Colab's /content folder.
            uploaded_path = Path('/content') / uploaded_name
            # Save the uploaded bytes as a real file so Python can open it normally.
            uploaded_path.write_bytes(uploaded_bytes)
            # Check whether the uploaded file is a ZIP archive.
            if zipfile.is_zipfile(uploaded_path):
                # Open the ZIP archive when the uploaded file is a ZIP file.
                with zipfile.ZipFile(uploaded_path, 'r') as zip_object:
                    # Extract every file from the ZIP archive into /content.
                    zip_object.extractall('/content')
    # Handle the case where the notebook is not running inside Google Colab.
    except ImportError:
        # Stop the notebook with a clear message because interactive Colab upload is unavailable.
        raise FileNotFoundError(f'Missing files: {missing_files}')

# Search again after the optional upload and store the final location of every tutorial file.
paths = {short_name: find_file(filename) for short_name, filename in EXPECTED_FILES.items()}

# Build a second list to confirm that nothing is still missing.
still_missing = [EXPECTED_FILES[name] for name, path in paths.items() if path is None]

# Stop with a clear error if any required file still cannot be found.
if still_missing:
    # Raise a FileNotFoundError that names exactly which tutorial files are still missing.
    raise FileNotFoundError(f'Could not find these files: {still_missing}')

# Print a heading so the file locations below are easy to recognize.
print('Files ready:')

# Print the short name and actual path for each file that Python will use.
for name, path in paths.items():
    # Show one readable line for the current dataset and its resolved file path.
    print(f'  {name:10s} -> {path}')

## A file-opening pattern you should remember

For a **fresh notebook**, the normal workflow is much simpler when the CSV is already in the Colab Files panel:

```python
import pandas as pd

df = pd.read_csv('my_file.csv')
```

The longer setup above is used only to make this tutorial convenient and robust. In your own work, you will often upload one file and read it directly with `pd.read_csv()`.

# Part 2 — Load, prepare, and inspect the data

Before making a graph, always inspect the data.

We want to answer these questions first:

- What are the column names?
- How many rows and columns do we have?
- Which columns are numeric?
- Are dates stored as real dates or just text?
- Are there missing values?
- Is the time series in chronological order?

A graph can be misleading if the dates are not recognized correctly or the observations are out of order.

In [ ]:
# Read the insurance CSV file into a pandas DataFrame called insurance.
insurance = pd.read_csv(paths['insurance'])

# Read the hourly weather CSV file into a pandas DataFrame called weather.
weather = pd.read_csv(paths['weather'])

# Read the Apple stock CSV file into a pandas DataFrame called apple.
apple = pd.read_csv(paths['apple'])

# Read the quarterly arrivals CSV file into a pandas DataFrame called arrivals.
arrivals = pd.read_csv(paths['arrivals'])

# Read the annual passenger CSV file into a pandas DataFrame called passengers.
passengers = pd.read_csv(paths['passengers'])

# Convert the weather date-and-time text into real pandas datetime values using its known format.
weather['datetime_local'] = pd.to_datetime(weather['datetime_local'], format='%d-%m-%Y %H:%M')

# Convert the Apple Date column from text into real pandas datetime values.
apple['Date'] = pd.to_datetime(apple['Date'])

# Convert the arrivals Date column from text into real pandas datetime values.
arrivals['Date'] = pd.to_datetime(arrivals['Date'])

# Convert the insurance Month column from text such as '2002 Jan' into real monthly dates.
insurance['Month'] = pd.to_datetime(insurance['Month'], format='%Y %b')

# Sort weather observations from earliest to latest and rebuild the row numbers.
weather = weather.sort_values('datetime_local').reset_index(drop=True)

# Sort Apple observations from earliest to latest and rebuild the row numbers.
apple = apple.sort_values('Date').reset_index(drop=True)

# Sort arrivals first by country and then by date so every country's series is chronological.
arrivals = arrivals.sort_values(['Country', 'Date']).reset_index(drop=True)

# Sort insurance observations by month and rebuild the row numbers.
insurance = insurance.sort_values('Month').reset_index(drop=True)

# Sort passenger observations by year and rebuild the row numbers.
passengers = passengers.sort_values('Year').reset_index(drop=True)

# Print a confirmation message after all five datasets have been loaded and prepared.
print('All datasets loaded and date columns prepared.')

### Why convert dates with `pd.to_datetime()`?

A date such as `2020-09-21` may look like a date to us, but a CSV file can store it as ordinary text.

After `pd.to_datetime()` converts the column, Python understands chronological operations such as:

- sorting observations correctly;
- extracting year, month, quarter, day, or hour;
- placing dates properly on a graph;
- setting a time-series frequency;
- performing time-series decomposition.

**Course habit:** if a variable represents time, check that Python recognizes it as a date/time variable before doing time-series analysis.

In [ ]:
# Create a dictionary so we can inspect all datasets with one short loop.
datasets = {'insurance': insurance, 'weather': weather, 'apple': apple, 'arrivals': arrivals, 'passengers': passengers}

# Loop through each dataset name and DataFrame in the dictionary.
for name, df in datasets.items():
    # Print a visible separator and the dataset name.
    print(f'\n--- {name.upper()} ---')
    # Print the number of rows and columns in the dataset.
    print('Shape:', df.shape)
    # Print the first three rows so we can see actual values and column names.
    print(df.head(3).to_string(index=False))
    # Print the total number of missing values across the entire dataset.
    print('Total missing values:', int(df.isna().sum().sum()))

## What to look for when inspecting data

`df.shape` tells us the size of the table as `(rows, columns)`.

`df.head()` lets us visually check whether variables were read correctly.

`df.isna().sum()` checks for missing values. Missing observations matter because many time-series methods require a complete sequence of periods.

Later, when you begin your own analysis, this simple inspection step can save a great deal of debugging time.

# Part 3 — Scatter plots

A scatter plot studies the relationship between **two quantitative variables**.

Each dot represents one observation. The horizontal variable is usually called `x`, and the vertical variable is usually called `y`.

When you read a scatter plot, look for five things:

1. **Direction** — positive, negative, or no clear direction.
2. **Strength** — tightly clustered pattern or widely scattered points.
3. **Form** — approximately linear, curved, or something more complex.
4. **Outliers** — unusual points far from the main pattern.
5. **Groups/clusters** — separate collections of observations that may behave differently.

A scatter plot shows **association**, not automatic proof of causation.

## 3.1 Simple scatter plot — Insurance advertising and quotes

### Question
Do months with more TV advertising tend to have more insurance quotes?

We will place `TVadverts` on the horizontal axis and `Quotes` on the vertical axis. We will also calculate the correlation coefficient to summarize the linear relationship numerically.

In [ ]:
# Create one figure and one plotting area; ax is the plotting area we will modify.
fig, ax = plt.subplots(figsize=(8, 5.5))

# Draw one point for every month using TV advertising as x and insurance quotes as y.
ax.scatter(insurance['TVadverts'], insurance['Quotes'], alpha=0.8)

# Add a descriptive title that tells the reader what relationship is being displayed.
ax.set_title('Insurance Quotes vs TV Advertising')

# Label the horizontal axis with the predictor/explanatory variable shown on x.
ax.set_xlabel('TV advertising')

# Label the vertical axis with the response variable shown on y.
ax.set_ylabel('Insurance quotes')

# Adjust spacing so labels and titles fit neatly inside the figure.
plt.tight_layout()

# Display the completed scatter plot.
plt.show()

# Calculate Pearson's correlation between TV advertising and insurance quotes.
corr = insurance['TVadverts'].corr(insurance['Quotes'])

# Print the correlation rounded to three decimal places for easy interpretation.
print(f'Correlation between TV advertising and quotes: {corr:.3f}')

### Interpretation

The points form a strong upward pattern. The correlation is approximately **0.93**, which is close to `+1`.

That tells us:

- months with higher TV advertising tend to have higher insurance quote activity;
- the relationship is strongly positive and approximately linear in this sample;
- the points are not perfectly on one straight line, so advertising is not the only source of variation.

What it **does not** tell us is that TV advertising alone caused the quote changes. Other variables can change at the same time.

### What you should remember
A scatter plot is usually one of the first tools to use when you want to understand whether two numeric variables move together.

## 3.2 Grouped scatter plot — Temperature and dew point by day/night

Sometimes an overall relationship hides differences between categories.

Here we create a new categorical variable called `DayPeriod`:

- hours from 7:00 through 18:59 are labeled **Day**;
- the remaining hours are labeled **Night**.

Then we plot temperature against dew point separately for the two groups.

### Why group a scatter plot?
Grouping helps us ask whether two categories have:

- different levels;
- different relationships;
- separate clusters;
- different slopes or spreads.

In [ ]:
# Extract the hour number from each weather datetime so we can classify observations by time of day.
weather['hour'] = weather['datetime_local'].dt.hour

# Create DayPeriod: label hours 7 through 18 as Day and all remaining hours as Night.
weather['DayPeriod'] = np.where(weather['hour'].between(7, 18), 'Day', 'Night')

# Create one figure and one plotting area for the grouped scatter plot.
fig, ax = plt.subplots(figsize=(8.5, 5.5))

# Split the weather data into Day and Night groups and process each group separately.
for group, group_df in weather.groupby('DayPeriod'):
    # Plot temperature against dew point for the current group and add its group name to the legend.
    ax.scatter(group_df['temperature'], group_df['dew_point'], alpha=0.65, label=group)

# Add a title describing the variables and the grouping variable.
ax.set_title('Temperature vs Dew Point — Grouped by Day/Night')

# Label the horizontal axis as temperature.
ax.set_xlabel('Temperature')

# Label the vertical axis as dew point.
ax.set_ylabel('Dew point')

# Add a legend so the Day and Night points can be distinguished.
ax.legend(title='Period')

# Adjust spacing so the graph elements do not overlap.
plt.tight_layout()

# Display the completed grouped scatter plot.
plt.show()

### Interpretation

The grouped graph gives us **two layers of information**:

1. the overall temperature–dew-point relationship;
2. whether daytime and nighttime observations occupy different regions of the graph.

If the two groups form very different clouds of points, the category may matter in later modeling. If they mostly overlap, the relationship may be similar across the two periods.

A grouped plot is therefore more informative than a single overall scatter plot whenever a meaningful category may change the relationship.

## 3.3 Scatterplot matrix — many pairwise relationships at once

A **scatterplot matrix** is useful when several numeric variables may be related.

Suppose we have five variables. Checking every possible pair one graph at a time would be slow. The scatterplot matrix displays the pairwise relationships in one figure.

### How to read the matrix

- **Rows and columns** correspond to variables.
- **Off-diagonal panels** are ordinary scatter plots comparing two variables.
- **Diagonal panels** show the distribution of each individual variable; here we use histograms.
- The matrix is **symmetric**: the relationship between temperature and wind speed appears twice, with the axes reversed.

### What we can learn from pairwise graphs

Pairwise plots help us identify:

- which variables move in the same direction;
- which variables move in opposite directions;
- whether relationships are strong or weak;
- whether the relationship is linear or curved;
- possible outliers;
- clusters or separate groups;
- variables that may contain very similar information.

That last point matters for modeling. If two potential predictors are extremely strongly related to each other, they may provide partly redundant information.

### Important limitation
A scatterplot matrix studies variables **two at a time**. It does not show the full effect of several variables acting together, and it does not prove causation.

In [ ]:
# Select only the five numeric weather variables we want to compare in the matrix.
weather_vars = weather[['temperature', 'dew_point', 'wind_speed', 'pressure', 'ozone']]

# Create the scatterplot matrix; diagonal='hist' places a histogram on each diagonal panel.
axes = scatter_matrix(weather_vars, figsize=(11, 11), diagonal='hist', alpha=0.55)

# Add one overall title above the full matrix and move it slightly upward for spacing.
plt.suptitle('Scatterplot Matrix — Selected Weather Variables', y=0.92)

# Display the full pairwise plot matrix.
plt.show()

### Reading this specific matrix

In this dataset, the selected weather variables do **not** show any extremely strong pairwise linear relationship.

A few patterns are still visible:

- temperature and dew point show a modest negative relationship;
- temperature and wind speed show a weak positive relationship;
- temperature and pressure show a modest negative relationship;
- several other relationships are weak and widely scattered.

This is useful information. It tells us that the variables are not simply duplicates of one another, but it also warns us that no single pairwise relationship explains everything.

A good next step is to calculate a **correlation matrix**, which gives a numeric summary of the same linear relationships.

In [ ]:
# Calculate Pearson correlations for every pair of the selected weather variables.
weather_corr = weather_vars.corr()

# Round the correlation values to three decimal places so the table is easy to read.
weather_corr = weather_corr.round(3)

# Print a heading before the correlation matrix.
print('Pairwise correlation matrix:')

# Display the completed correlation matrix.
print(weather_corr)

### How to connect the correlation table to the pairwise graphs

The sign of a correlation tells us the **direction**:

- positive correlation → the variables tend to rise together;
- negative correlation → one tends to be higher when the other is lower.

The absolute size gives a rough idea of **linear strength**:

- closer to `0` → weak linear relationship;
- closer to `1` or `-1` → stronger linear relationship.

But the scatter plot is still essential because correlation alone can hide curves, clusters, and outliers.

**Course habit:** use the graph and the number together rather than relying only on one of them.

# Part 4 — Line plots and time plots

A time plot places observations in chronological order along the horizontal axis and connects consecutive values.

This is one of the most important graphs in forecasting because it can reveal:

- **trend** — long-run upward or downward movement;
- **seasonality** — a repeating pattern at a fixed frequency;
- **cycles** — longer rises and falls that are not necessarily fixed in length;
- **outliers** — unusual individual observations;
- **structural changes** — sudden changes in level or behavior;
- **changing variability** — fluctuations becoming larger or smaller over time.

### Critical preparation step
Always sort the dataset by time before creating a time plot. Otherwise Python may connect observations in the wrong order.

## 4.1 Apple daily Open and Close prices

We plot two variables on the same time axis:

- `Open` — the opening price for the trading day;
- `Close` — the closing price for the trading day.

Because both variables use the same units and refer to the same stock, displaying them together is useful for comparison.

In [ ]:
# Create one figure and one plotting area for the two stock-price series.
fig, ax = plt.subplots(figsize=(11, 5.5))

# Plot the daily opening price against Date and label the line Open.
ax.plot(apple['Date'], apple['Open'], label='Open', linewidth=1.4)

# Plot the daily closing price against Date and label the line Close.
ax.plot(apple['Date'], apple['Close'], label='Close', linewidth=1.4)

# Add a descriptive title to the graph.
ax.set_title('Apple Daily Stock Prices')

# Label the horizontal axis as Date.
ax.set_xlabel('Date')

# Label the vertical axis as Price.
ax.set_ylabel('Price')

# Add a legend so the opening and closing series can be identified.
ax.legend()

# Adjust graph spacing so labels fit cleanly.
plt.tight_layout()

# Display the completed time plot.
plt.show()

# Print the first closing price to show the starting level of the sample.
print(f"First close: ${apple['Close'].iloc[0]:.2f}")

# Print the last closing price to show the ending level of the sample.
print(f"Last close:  ${apple['Close'].iloc[-1]:.2f}")

# Print the minimum closing price observed during the sample.
print(f"Minimum close in sample: ${apple['Close'].min():.2f}")

# Print the maximum closing price observed during the sample.
print(f"Maximum close in sample: ${apple['Close'].max():.2f}")

### Interpretation

The Open and Close lines move very closely together because they measure prices for the same stock on the same trading days.

The series contains substantial short-run variation. Over this particular sample, the ending level is above the beginning level, but there is no clear fixed seasonal pattern visible from the graph.

For forecasting, the key lesson is not “the stock went up.” The key lesson is to **look at the structure before choosing a model**. Financial price data often behave differently from strongly seasonal business series.

## 4.2 Grouped time plot — Quarterly arrivals by country

A grouped time plot displays several related time series on the same graph.

This lets us compare:

- levels;
- growth patterns;
- turning points;
- seasonal movements;
- relative variability.

Each country becomes one separate line.

In [ ]:
# Create one figure and one plotting area for all country arrival series.
fig, ax = plt.subplots(figsize=(11, 6))

# Split the data by Country so each country can be plotted as its own line.
for country, group_df in arrivals.groupby('Country'):
    # Plot the current country's arrivals over time and use the country name in the legend.
    ax.plot(group_df['Date'], group_df['arrival'], label=country, linewidth=1.4)

# Add a title describing the grouped time series.
ax.set_title('Quarterly Arrivals by Country')

# Label the horizontal axis as Date.
ax.set_xlabel('Date')

# Label the vertical axis as Arrivals.
ax.set_ylabel('Arrivals')

# Add a legend identifying each country line.
ax.legend(title='Country')

# Adjust spacing so graph elements fit neatly.
plt.tight_layout()

# Display the grouped time plot.
plt.show()

### How to interpret several time series together

Do not try to read every point individually. Start with broader questions:

1. Which series usually has the highest or lowest level?
2. Which series appears to grow or decline over time?
3. Do several countries experience peaks or drops around the same dates?
4. Does each country show a repeating quarterly pattern?
5. Does the size of the fluctuations grow as the series level grows?

These observations can later influence whether we model the countries separately, use seasonal methods, or investigate shared external drivers.

# Part 5 — Seasonal plots

A seasonal plot reorganizes a time series so that the observations are compared by **season within each cycle**.

For quarterly data:

- the seasonal period is `4`;
- Q1, Q2, Q3, and Q4 are the seasons;
- each line can represent one year.

A normal time plot answers: **“What happened over time?”**

A seasonal plot answers: **“Does the same part of the year repeatedly behave differently?”**

This makes repeated seasonal behavior much easier to recognize.

## 5.1 Seasonal plot — U.S. quarterly arrivals

We first isolate the U.S. observations, then create two helper variables:

- `Year` — used to draw a separate line for each year;
- `Quarter` — used on the horizontal axis from Q1 to Q4.

In [ ]:
# Select only rows where Country is US and make an independent copy of those rows.
us = arrivals.loc[arrivals['Country'] == 'US'].copy()

# Extract the calendar year from each U.S. date and store it in a new Year column.
us['Year'] = us['Date'].dt.year

# Extract the quarter number from each U.S. date and store it in a new Quarter column.
us['Quarter'] = us['Date'].dt.quarter

# Create one figure and one plotting area for the seasonal plot.
fig, ax = plt.subplots(figsize=(10.5, 6))

# Split the U.S. observations by year so every year becomes one line across quarters.
for year, group_df in us.groupby('Year'):
    # Plot quarter against arrivals for the current year and mark each quarterly observation with a point.
    ax.plot(group_df['Quarter'], group_df['arrival'], marker='o', alpha=0.7, label=str(year))

# Add a title describing the series and plot type.
ax.set_title('Seasonal Plot — U.S. Quarterly Arrivals')

# Label the horizontal axis as Quarter.
ax.set_xlabel('Quarter')

# Label the vertical axis as Arrivals.
ax.set_ylabel('Arrivals')

# Replace numeric quarter labels 1 through 4 with the more readable labels Q1 through Q4.
ax.set_xticks([1, 2, 3, 4], ['Q1', 'Q2', 'Q3', 'Q4'])

# Place the year legend outside the main graph because many years are displayed.
ax.legend(title='Year', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)

# Adjust spacing so the outside legend and graph labels fit properly.
plt.tight_layout()

# Display the completed seasonal plot.
plt.show()

# Calculate the average number of arrivals for each quarter across all U.S. years.
quarter_means = us.groupby('Quarter')['arrival'].mean()

# Print a heading for the quarterly averages.
print('Average U.S. arrivals by quarter:')

# Print the quarter averages rounded to two decimal places.
print(quarter_means.round(2))

### Interpretation

The quarterly averages show that Q1 and Q4 are highest on average, while Q2 is lower.

The important concept is **repetition**. If similar quarter-to-quarter differences recur over many years, the series contains a seasonal pattern.

Seasonality matters because a forecasting model that ignores a strong quarterly pattern may systematically forecast some quarters too high and others too low.

### Seasonal plot versus time plot

Use both:

- the **time plot** shows trend and historical movement;
- the **seasonal plot** isolates the within-year pattern.

# Part 6 — Moving averages

A moving average smooths a series by replacing an observation with an average of nearby observations.

The purpose is to reduce short-run fluctuations so that the underlying movement becomes easier to see.

### Main trade-off

- a **shorter window** reacts quickly but remains relatively noisy;
- a **longer window** is smoother but reacts more slowly.

The number of observations included in the average is called the **window** or **order** of the moving average.

## 6.1 Trailing moving averages — Apple 5-day and 15-day

A trailing moving average uses the current observation and earlier observations only.

For a 5-day moving average:

\[
MA_t(5)=\frac{y_t+y_{t-1}+y_{t-2}+y_{t-3}+y_{t-4}}{5}
\]

The first four rows cannot have a complete 5-day average because five observations are not available yet. Pandas therefore returns missing values for those beginning rows.

In [ ]:
# Calculate a trailing 5-day average of the Close price and store it in a new MA5 column.
apple['MA5'] = apple['Close'].rolling(window=5).mean()

# Calculate a trailing 15-day average of the Close price and store it in a new MA15 column.
apple['MA15'] = apple['Close'].rolling(window=15).mean()

# Create one figure and one plotting area for the raw and smoothed series.
fig, ax = plt.subplots(figsize=(11, 5.8))

# Plot the original closing price with partial transparency so the smoother lines remain easy to see.
ax.plot(apple['Date'], apple['Close'], label='Close', alpha=0.55, linewidth=1)

# Plot the 5-day moving average to show a relatively responsive smoothed line.
ax.plot(apple['Date'], apple['MA5'], label='5-day moving average', linewidth=1.8)

# Plot the 15-day moving average to show a smoother but slower-moving line.
ax.plot(apple['Date'], apple['MA15'], label='15-day moving average', linewidth=2)

# Add a descriptive graph title.
ax.set_title('Apple Close Price with 5-Day and 15-Day Moving Averages')

# Label the horizontal axis as Date.
ax.set_xlabel('Date')

# Label the vertical axis as Price.
ax.set_ylabel('Price')

# Add a legend identifying the original series and both moving averages.
ax.legend()

# Adjust spacing so graph elements fit correctly.
plt.tight_layout()

# Display the completed moving-average plot.
plt.show()

### Interpretation

The 5-day average follows recent changes more closely. The 15-day average removes more short-run movement and therefore looks smoother.

However, smoothness has a cost: the 15-day average reacts later when the series changes direction.

This is called **lag**. Increasing the moving-average window generally increases smoothness and lag.

### Important coding pattern

```python
df['MA5'] = df['value'].rolling(window=5).mean()
```

Read this as: “Create a new column by taking a rolling five-observation mean of the value column.”

## 6.2 Centered moving average

A centered moving average uses observations on **both sides** of the target time point.

For an odd window of 5:

\[
CMA_t(5)=\frac{y_{t-2}+y_{t-1}+y_t+y_{t+1}+y_{t+2}}{5}
\]

Because future observations are needed, a centered moving average is useful for **describing trend-cycle movement in historical data**, but it cannot be calculated at the very end of a series in real time until future values become available.

In [ ]:
# Calculate a centered 5-day moving average by setting center=True inside rolling().
apple['CMA5'] = apple['Close'].rolling(window=5, center=True).mean()

# Create one figure and one plotting area for the original and centered-average series.
fig, ax = plt.subplots(figsize=(11, 5.5))

# Plot the original Close series with transparency so the smoother centered line stands out.
ax.plot(apple['Date'], apple['Close'], label='Close', alpha=0.5, linewidth=1)

# Plot the centered 5-day moving average as the smoother trend-like line.
ax.plot(apple['Date'], apple['CMA5'], label='Centered 5-day MA', linewidth=2)

# Add a title identifying the smoothing method.
ax.set_title('Centered 5-Day Moving Average')

# Label the horizontal axis as Date.
ax.set_xlabel('Date')

# Label the vertical axis as Price.
ax.set_ylabel('Price')

# Add a legend identifying the two lines.
ax.legend()

# Adjust graph spacing.
plt.tight_layout()

# Display the completed graph.
plt.show()

### Trailing versus centered moving average

A **trailing** moving average is backward-looking: it uses the current and earlier observations.

A **centered** moving average is descriptive: it places the average at the middle of the observations used.

Centered averages are especially important when we want to estimate a historical trend-cycle for decomposition.

## 6.3 Weighted moving average

A simple moving average gives every observation the same weight.

A weighted moving average allows some observations to matter more than others.

For example:

\[
WMA_t=0.1y_{t-2}+0.2y_{t-1}+0.7y_t
\]

The weights add to `1.0`, and the newest observation receives the largest weight. This makes the smoothed series more responsive to recent movement.

In [ ]:
# Create a NumPy array containing the three weights from oldest to newest observation.
weights = np.array([0.1, 0.2, 0.7])

# Start a 3-observation rolling window on the Close column.
rolling_close = apple['Close'].rolling(window=3)

# Apply the weighted-average formula to each complete 3-observation window and store the result in WMA3.
apple['WMA3'] = rolling_close.apply(lambda x: np.dot(x, weights), raw=True)

# Create one figure and one plotting area for the raw and weighted-average series.
fig, ax = plt.subplots(figsize=(11, 5.5))

# Plot the original closing price with transparency for comparison.
ax.plot(apple['Date'], apple['Close'], label='Close', alpha=0.45, linewidth=1)

# Plot the weighted moving average so we can see how the heavier recent weight changes responsiveness.
ax.plot(apple['Date'], apple['WMA3'], label='Weighted MA (0.1, 0.2, 0.7)', linewidth=2)

# Add a title identifying the method.
ax.set_title('Weighted Moving Average')

# Label the horizontal axis as Date.
ax.set_xlabel('Date')

# Label the vertical axis as Price.
ax.set_ylabel('Price')

# Add a legend identifying the original and weighted-average lines.
ax.legend()

# Adjust figure spacing.
plt.tight_layout()

# Display the completed weighted moving-average graph.
plt.show()

### Why use weights?

Weights allow the analyst to express that some observations are more relevant than others.

With weights `0.1, 0.2, 0.7`, the newest value has seven times the influence of the oldest value in the three-period window.

The choice of weights should have a reason. We should not choose them only because a graph “looks good.”

## 6.4 Even seasonal periods and the `2×m` moving average

Quarterly data have a seasonal period of `m = 4`, which is even.

A 4-term moving average falls mathematically **between** two quarterly observations rather than directly on a quarter. To center the estimate on an actual time point, we average two adjacent 4-term moving averages.

This is called a **2×4 moving average**.

The same idea applies to other even seasonal periods. For example, monthly data with annual seasonality use an even period of `12`, so a `2×12` moving average can be used to estimate the trend-cycle.

In [ ]:
# Move the U.S. arrival values into a time-indexed pandas Series using Date as the index.
us_series = us.set_index('Date')['arrival'].sort_index()

# Tell pandas that the expected U.S. observations occur at the start of each calendar quarter.
us_series = us_series.asfreq('QS-JAN')

# Calculate the first-stage trailing 4-quarter moving average.
ma4 = us_series.rolling(window=4).mean()

# Average each pair of adjacent 4-quarter averages and shift the result so it aligns with the true center.
two_by_four_ma = ma4.rolling(window=2).mean().shift(-2)

# Create one figure and one plotting area for the quarterly series and its trend estimate.
fig, ax = plt.subplots(figsize=(11, 5.8))

# Plot the original quarterly arrival series with transparency.
ax.plot(us_series.index, us_series, label='U.S. arrivals', alpha=0.5, linewidth=1)

# Plot the centered 2×4 moving average as an estimate of the smoother trend-cycle.
ax.plot(two_by_four_ma.index, two_by_four_ma, label='2×4 moving average', linewidth=2.2)

# Add a title explaining the series and smoothing method.
ax.set_title('U.S. Quarterly Arrivals with 2×4 Moving Average')

# Label the horizontal axis as Date.
ax.set_xlabel('Date')

# Label the vertical axis as Arrivals.
ax.set_ylabel('Arrivals')

# Add a legend identifying the original and smoothed lines.
ax.legend()

# Adjust spacing so the graph is easy to read.
plt.tight_layout()

# Display the completed 2×4 moving-average graph.
plt.show()

### Interpretation

The `2×4` moving average removes much of the repeating quarterly movement and shows the underlying longer-run direction more clearly.

This is a key conceptual link between **moving averages** and **decomposition**: moving averages can help estimate the trend-cycle component of a seasonal time series.

# Part 7 — Time-series decomposition

Decomposition separates an observed time series into simpler components.

We usually focus on three components:

- **Trend-cycle (`T`)** — the longer-run underlying movement;
- **Seasonal (`S`)** — a repeating pattern tied to the calendar/frequency;
- **Remainder (`R`)** — the irregular part left after trend and seasonality are accounted for.

## Additive decomposition

\[
y_t=T_t+S_t+R_t
\]

Use the additive form when seasonal fluctuations are roughly the **same absolute size** across the series.

Example idea: a business may repeatedly sell about 200 extra units every December, regardless of whether its overall level is high or low.

## Multiplicative decomposition

\[
y_t=T_t\times S_t\times R_t
\]

Use the multiplicative form when seasonal fluctuations are better described as **percentages or ratios** of the series level.

Example idea: December sales may repeatedly be about 20% above the underlying level. As the business grows, the absolute seasonal swing also grows.

### Before choosing
Always graph the series first. Decomposition should follow the structure visible in the data rather than being chosen automatically.

## 7.1 Multiplicative decomposition — U.S. quarterly arrivals

The U.S. arrival series is positive and quarterly, so the seasonal period is `4`.

The `seasonal_decompose()` function will return:

- `observed` — the original series;
- `trend` — estimated trend-cycle;
- `seasonal` — repeating quarterly seasonal factors;
- `resid` — remainder/residual component.

Before decomposition, we verify that the quarterly series does not contain missing periods.

In [ ]:
# Print the number of quarterly observations available in the U.S. series.
print('Number of quarterly observations:', len(us_series))

# Count and print missing quarterly values because decomposition requires a complete regular series.
print('Missing observations:', int(us_series.isna().sum()))

# Decompose the U.S. series using a multiplicative model with four quarters in each seasonal cycle.
us_decomp = seasonal_decompose(us_series, model='multiplicative', period=4, extrapolate_trend='freq')

# Ask statsmodels to create the standard four-panel decomposition figure.
fig = us_decomp.plot()

# Increase the figure size so the four decomposition components are easy to read.
fig.set_size_inches(11, 9)

# Add one overall title above the decomposition panels.
fig.suptitle('Multiplicative Decomposition — U.S. Quarterly Arrivals', y=1.01)

# Adjust spacing between graph elements.
plt.tight_layout()

# Display the completed decomposition plot.
plt.show()

### How to read the four decomposition panels

**Observed:** the original data we started with.

**Trend:** the smoother long-run movement after much of the seasonal fluctuation has been removed.

**Seasonal:** the repeating quarterly effect. Because this is a multiplicative model, the seasonal component is expressed as a factor around `1`.

**Residual:** what remains after the trend and seasonal pattern have been accounted for. Values around `1` indicate the observation is close to what the trend and seasonal pattern together would imply.

A decomposition is useful because it lets us study different sources of movement separately instead of treating the observed series as one unexplained line.

## 7.2 Interpreting multiplicative seasonal factors

For a multiplicative model:

- seasonal factor `1.10` means about **10% above** trend;
- seasonal factor `0.90` means about **10% below** trend;
- seasonal factor `1.00` means approximately **no seasonal adjustment** relative to trend.

The next code cell averages the estimated seasonal factors by quarter so they are easier to interpret.

In [ ]:
# Group the estimated seasonal component by calendar quarter and calculate its average factor for each quarter.
seasonal_by_quarter = us_decomp.seasonal.groupby(us_decomp.seasonal.index.quarter).mean()

# Print a heading before the seasonal-factor interpretation.
print('Average multiplicative seasonal factors:')

# Loop through each quarter number and its average seasonal factor.
for quarter, factor in seasonal_by_quarter.items():
    # Convert the factor's distance from 1 into a percentage and print a business-friendly interpretation.
    print(f'Q{quarter}: {factor:.3f}  ({(factor - 1) * 100:+.1f}% relative to trend)')

### Interpretation of this series

For this U.S. arrival series, the estimated factors are approximately:

- Q1: `1.081` → about **8.1% above** trend;
- Q2: `0.906` → about **9.4% below** trend;
- Q3: `0.937` → about **6.3% below** trend;
- Q4: `1.076` → about **7.6% above** trend.

These numbers translate the seasonal graph into a quantitative seasonal effect.

Notice how the result agrees with the earlier seasonal plot: Q1 and Q4 are relatively strong, while Q2 and Q3 are weaker.

## 7.3 Seasonally adjusted series

A seasonally adjusted series removes the regular seasonal effect so periods can be compared more fairly.

For a multiplicative model:

\[
SA_t=\frac{y_t}{S_t}
\]

For example, if a quarter is normally 8% above trend, seasonal adjustment removes that predictable seasonal boost.

This helps us see whether a change is due to the underlying movement rather than simply the normal time of year.

In [ ]:
# Remove the multiplicative seasonal effect by dividing observed arrivals by the estimated seasonal factor.
seasonally_adjusted = us_series / us_decomp.seasonal

# Create one figure and one plotting area for the observed and seasonally adjusted series.
fig, ax = plt.subplots(figsize=(11, 5.5))

# Plot the original U.S. arrival series with transparency for comparison.
ax.plot(us_series.index, us_series, label='Observed', alpha=0.55, linewidth=1.2)

# Plot the seasonally adjusted series so the underlying movement is easier to compare across quarters.
ax.plot(seasonally_adjusted.index, seasonally_adjusted, label='Seasonally adjusted', linewidth=2)

# Add a descriptive graph title.
ax.set_title('U.S. Arrivals — Observed vs Seasonally Adjusted')

# Label the horizontal axis as Date.
ax.set_xlabel('Date')

# Label the vertical axis as Arrivals.
ax.set_ylabel('Arrivals')

# Add a legend identifying the original and adjusted series.
ax.legend()

# Adjust graph spacing.
plt.tight_layout()

# Display the comparison graph.
plt.show()

### Why seasonal adjustment matters

Suppose Q4 is normally a strong quarter. A raw Q4 value may look unusually high compared with Q3 even when nothing unusual happened.

Seasonal adjustment asks a better question:

> “After removing the normal seasonal effect, is the underlying level actually higher or lower?”

This is why many economic indicators are reported in seasonally adjusted form.

# Part 8 — Additive or multiplicative? A practical decision guide

Use an **additive** decomposition when:

- seasonal swings are roughly constant in absolute size;
- a peak may be about the same number of units above trend at low and high levels;
- variability does not noticeably expand as the level rises.

Use a **multiplicative** decomposition when:

- seasonal movement is better interpreted as a percentage or ratio;
- higher series levels tend to produce larger absolute seasonal swings;
- observations are positive.

### A visual rule of thumb
If the seasonal “waves” appear to get larger as the series level rises, multiplicative decomposition may be more appropriate.

If the seasonal waves remain roughly the same size, additive decomposition may be more appropriate.

This is a judgment based on the data, not a rule that should be applied blindly.

# Part 9 — What you should be able to write in a fresh notebook

This section is deliberately practical. These are the pieces of Python you will repeatedly need in later tutorials and assignments.

You do **not** need to memorize every plotting option. You should, however, become comfortable with the structure of the workflow:

1. import packages;
2. upload/open the file;
3. inspect the data;
4. convert the date column;
5. sort by date;
6. create the graph or transformation;
7. interpret the result.

## 9.1 Fresh notebook starter — imports

Copy this near the top of a new notebook when you need standard data and graph tools:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
```

### Remember what each one does

- `np` → numerical operations;
- `pd` → data tables, CSV files, dates, rolling calculations;
- `plt` → graphs;
- `seasonal_decompose` → time-series decomposition.

## 9.2 Fresh notebook starter — opening a CSV file in Colab

### Method A: the file is already visible in the Colab Files panel

```python
df = pd.read_csv('my_data.csv')
```

### Method B: ask the user to upload a file

```python
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('my_data.csv')
```

### Immediately check the file

```python
print(df.head())
print(df.columns)
print(df.shape)
print(df.isna().sum())
```

This quickly tells you whether the file was read as expected.

## 9.3 Fresh time-series starter — date preparation

If your date column is called `Date`:

```python
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
```

If Python cannot automatically understand the date format, provide the format explicitly. For example:

```python
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
```

### Why these two lines matter

`pd.to_datetime()` tells Python that the column represents time.

`sort_values()` guarantees that observations are in chronological order before we connect them with a line.

## 9.4 Fresh plotting templates

### Scatter plot

```python
plt.scatter(df['x_variable'], df['y_variable'])
plt.xlabel('X variable')
plt.ylabel('Y variable')
plt.title('Y vs X')
plt.show()
```

### Time plot

```python
plt.plot(df['Date'], df['value'])
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Time Plot')
plt.show()
```

### Moving average

```python
df['MA5'] = df['value'].rolling(window=5).mean()
```

### Centered moving average

```python
df['CMA5'] = df['value'].rolling(window=5, center=True).mean()
```

The variable names will change from dataset to dataset, but the structure is reusable.

## 9.5 Fresh decomposition template

For a regular quarterly pandas Series called `series`:

```python
result = seasonal_decompose(
    series,
    model='multiplicative',
    period=4
)

result.plot()
plt.show()
```

### What must be correct before this runs

- `series` must be in chronological order;
- the time frequency must be regular;
- there should not be missing seasonal periods;
- `period=4` is correct for quarterly seasonality, but another frequency needs another period;
- multiplicative decomposition requires positive observations.

## 9.6 Tested fresh-workflow example

The next cell deliberately starts again with the Apple CSV and repeats the essential workflow in compact form.

This is the type of code you should eventually be comfortable writing yourself in a new notebook.

In [ ]:
# Import pandas for reading and manipulating the CSV data.
import pandas as pd

# Import Matplotlib for creating the graph.
import matplotlib.pyplot as plt

# Read the Apple CSV file into a fresh DataFrame called fresh_df.
fresh_df = pd.read_csv(paths['apple'])

# Convert the Date column from text into real datetime values.
fresh_df['Date'] = pd.to_datetime(fresh_df['Date'])

# Sort the rows from earliest date to latest date and rebuild the row numbers.
fresh_df = fresh_df.sort_values('Date').reset_index(drop=True)

# Calculate a 10-day trailing moving average of the closing price.
fresh_df['MA10'] = fresh_df['Close'].rolling(window=10).mean()

# Create one figure and plotting area for the original series and moving average.
fig, ax = plt.subplots(figsize=(10, 5))

# Plot the original daily closing price.
ax.plot(fresh_df['Date'], fresh_df['Close'], label='Close', alpha=0.5)

# Plot the 10-day moving average on the same time axis.
ax.plot(fresh_df['Date'], fresh_df['MA10'], label='10-day MA', linewidth=2)

# Add a title describing what the graph contains.
ax.set_title('Fresh Workflow Example — Apple Close and 10-Day Moving Average')

# Label the horizontal axis.
ax.set_xlabel('Date')

# Label the vertical axis.
ax.set_ylabel('Price')

# Add a legend so the two lines can be identified.
ax.legend()

# Adjust spacing so the figure is easy to read.
plt.tight_layout()

# Display the completed graph.
plt.show()

# Part 10 — Practice

Try these without copying the earlier answer immediately.

### Question 1 — Scatter plot
Create a scatter plot of `wind_speed` versus `temperature` using the weather data.

Explain:
- the direction of the relationship;
- whether it looks strong or weak;
- whether any unusual observations are visible.

### Question 2 — Time plot
Plot `Passengers` against `Year` using `aus_airpassengers.csv`.

Explain the long-run pattern.

### Question 3 — Moving averages
Change the Apple moving-average windows from 5 and 15 days to 10 and 30 days.

Explain:
- which line is smoother;
- which line responds faster;
- why the difference occurs.

### Question 4 — Seasonal plot
Repeat the quarterly seasonal plot using `UK` instead of `US`.

Identify the strongest and weakest quarters.

### Question 5 — Decomposition
Create a quarterly decomposition for the U.K. arrival series using `period=4`.

Interpret the seasonal factors in plain business language.

# Final reference — the workflow to remember

When you receive a new time-series dataset, use this order:

**1. Import packages**  
Know which tools you need.

**2. Read the file**  
Use `pd.read_csv()` for a CSV file.

**3. Inspect the data**  
Use `head()`, `shape`, column names, and missing-value checks.

**4. Convert the time variable**  
Use `pd.to_datetime()` when appropriate.

**5. Sort chronologically**  
Never assume a CSV is already in correct time order.

**6. Graph first**  
Look for trend, seasonality, unusual values, changing variability, and relationships between variables.

**7. Transform or decompose when useful**  
Moving averages can reveal underlying movement; decomposition separates trend, seasonality, and remainder.

**8. Interpret in words**  
A correct graph without interpretation is incomplete analysis.

> **Business Forecasting habit:** understand the pattern first, then choose the forecasting method.